# CSCI 3202, Spring 2026
## Project Intermediate Report: Mancala AI
**Name:** Jeffrey Allen

---

## Project Status & Progress Report
**Current Status:** The core Mancala game engine is fully functional, incorporating all standard rules, including stone distribution, skipping the opponent's mancala, and the capture logic for landing in an empty pit on one's own side. The valid_move and winning_eval functions are implemented to handle game termination and scoring.

**Completed Tasks:**
* Implementation of the Mancala class and core game logic.
* Implementation of a random_move_generator for baseline testing.
* Validation of game rules via manual move sequences.
* Execution of a 100-game simulation between two random players to gather baseline statistics.

**Next Steps:**
* Implement a Minimax agent with Alpha-Beta pruning to replace the random player.
* Develop and test heuristic functions for the AI agent.

**AI Disclosure:**
While the core logic and software design of this Mancala engine are my own, I utilized Gemini to assist in translating that logic from C/C++ that I am very familiar with into specific Python 3 syntax.

In [1]:
import random
import numpy as np

# Same seed, so endgame is consistent
random.seed(109)

In [2]:
class Mancala:
    def __init__(self, pits_per_player=6, stones_per_pit=4):
        self.pits_per_player = pits_per_player
        # The board is a single list. Indices 0-5 are P1's pits, 6 is P1's goal.
        # Indices 7-12 are P2's pits, 13 is P2's goal.
        self.board = [stones_per_pit] * ((pits_per_player + 1) * 2)
        self.current_player = 1
        self.moves = []
        # Setting up records for the indices to move throuhg list
        self.p1_pits_index = [0, self.pits_per_player - 1]
        self.p1_mancala_index = self.pits_per_player
        self.p2_pits_index = [self.pits_per_player + 1, len(self.board) - 2]
        self.p2_mancala_index = len(self.board) - 1
        
        self.board[self.p1_mancala_index] = 0
        self.board[self.p2_mancala_index] = 0

    def display_board(self):
        #Didn't really use this in the actual final printing because I didn't want to print 100 games, but it was helpful in debugging
        player_1_pits = self.board[self.p1_pits_index[0]: self.p1_pits_index[1] + 1]
        player_1_mancala = self.board[self.p1_mancala_index]
        player_2_pits = self.board[self.p2_pits_index[0]: self.p2_pits_index[1] + 1]
        player_2_mancala = self.board[self.p2_mancala_index]

        print('P1               P2')
        print('     ____{}____     '.format(player_2_mancala))
        for i in range(self.pits_per_player):
            print('{} -> | {} | {} | <- {}'.format(i + 1, player_1_pits[i], player_2_pits[-(i + 1)], self.pits_per_player - i))
        print('         {}         '.format(player_1_mancala))
        print('Turn: P' + str(self.current_player))
        
    def valid_move(self, pit):
        #just a check incase the random selection tries to do something illegal
        if not (1 <= pit <= self.pits_per_player):
            return False
        index = (pit - 1) if self.current_player == 1 else (self.pits_per_player + pit)
        return self.board[index] > 0
        
    def random_move_generator(self):
        valid_pits = [p for p in range(1, self.pits_per_player + 1) if self.valid_move(p)]
        return random.choice(valid_pits) if valid_pits else None
    
    def play(self, pit, silent=False):
        
        if not self.valid_move(pit):
            return self.board
        # Convert the player's choice into list
        index = (pit - 1) if self.current_player == 1 else (self.pits_per_player + pit)
        stones = self.board[index]
        self.board[index] = 0
        self.moves.append((self.current_player, pit))

        curr_index = index
        while stones > 0:
            # Move to the next spot, going back to 0 if hit end (God bless modulo operation, thanks discrete math)
            curr_index = (curr_index + 1) % len(self.board)
            # Skip the opponent's goal and deposit
            if (self.current_player == 1 and curr_index == self.p2_mancala_index) or \
               (self.current_player == 2 and curr_index == self.p1_mancala_index):
                continue
            self.board[curr_index] += 1
            stones -= 1

        # Capture logic
        if self.board[curr_index] == 1:
            if ((self.current_player == 1 and self.p1_pits_index[0]) <= curr_index <= (self.p1_pits_index[1])) or \
               ((self.current_player == 2 and self.p2_pits_index[0]) <= curr_index <= (self.p2_pits_index[1])):
                opp_index = (2 * self.pits_per_player) - curr_index
                if self.board[opp_index] > 0:
                    captured = self.board[curr_index] + self.board[opp_index]
                    self.board[curr_index] = 0
                    self.board[opp_index] = 0
                    mancala_idx = self.p1_mancala_index if self.current_player == 1 else self.p2_mancala_index
                    self.board[mancala_idx] += captured

        self.current_player = 2 if self.current_player == 1 else 1 
        return self.board
    
    def winning_eval(self, silent=False):
        p1_stones = sum(self.board[self.p1_pits_index[0]: self.p1_pits_index[1] + 1])
        p2_stones = sum(self.board[self.p2_pits_index[0]: self.p2_pits_index[1] + 1])
        
        if p1_stones == 0 or p2_stones == 0:
            self.board[self.p1_mancala_index] += p1_stones
            self.board[self.p2_mancala_index] += p2_stones
            for i in range(len(self.board)):
                if i != self.p1_mancala_index and i != self.p2_mancala_index:
                    self.board[i] = 0
            return True
        return False

In [3]:
# Part 2: 100 Games (Random vs. Random)
num_games = 100
results = {"P1_wins": 0, "P2_wins": 0, "Ties": 0, "Total_Turns": []}

for _ in range(num_games): # the '_' is just shorthand for using i in C, just a trick
    game = Mancala()
    turns = 0
    while not game.winning_eval(): 
        move = game.random_move_generator()
        if move is None: break # just seeing if any actual moves can be made
        game.play(move, silent=True) #keeping silent
        turns += 1
    
    results["Total_Turns"].append(turns)
    p1_score = game.board[game.p1_mancala_index]
    p2_score = game.board[game.p2_mancala_index]
    
    if p1_score > p2_score:
        results["P1_wins"] += 1
    elif p2_score > p1_score:
        results["P2_wins"] += 1
    else:
        results["Ties"] += 1

# Report Statistics
print(f"Results for {num_games} Games:")
print(f"Player 1: Wins: {results['P1_wins']/num_games:.1%}, "
      f"Losses: {results['P2_wins']/num_games:.1%}, "
      f"Ties: {results['Ties']/num_games:.1%}")
print(f"Player 2: Wins: {results['P2_wins']/num_games:.1%}, "
      f"Losses: {results['P1_wins']/num_games:.1%}, "
      f"Ties: {results['Ties']/num_games:.1%}")
print(f"Average turns per game: {np.mean(results['Total_Turns']):.2f}")

Results for 100 Games:
Player 1: Wins: 43.0%, Losses: 49.0%, Ties: 8.0%
Player 2: Wins: 49.0%, Losses: 43.0%, Ties: 8.0%
Average turns per game: 32.57


## Intermediate Analysis & Conclusion

**1. Statistics Summary:**
* **Player 1 Win Rate:** 43%
* **Player 2 Win Rate:** 49%
* **Average Turns:** 32.57

**2. Is there a first-move advantage?**
Based on the results of 100 random vs. random games, there is no advantage as shown by Player 1 actually losing more than Player 2 in the game output.